## DEEP LEARNING


In [ ]:
#! pip install torch torchvision

   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/216.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/216.1 MB 981.3 kB/s eta 0:03:40
   ---------------------------------------- 1.3/216.1 MB 1.9 MB/s eta 0:01:54
    --------------------------------------- 3.1/216.1 MB 3.6 MB/s eta 0:01:00
   - -------------------------------------- 6.0/216.1 MB 5.7 MB/s eta 0:00:37
   -- ------------------------------------- 11.8/216.1 MB 9.4 MB/s eta 0:00:22
   --- ------------------------------------ 17.6/216.1 MB 12.1 MB/s eta 0:00:17
   ---- ----------------------------------- 23.1/216.1 MB 14.3 MB/s eta 0:00:14
   ---- ----------------------------------- 23.1/216.1 MB 14.3 MB/s eta 0:00:14
   ----- ---------------------------------- 28.8/216.1 MB 13.6 MB/s eta 0:00:14
   


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim


## NEURAL NETWORK FOR AND GATE


In [3]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [0.], [0.], [1.]])

In [4]:
#model definition
class ANDGateModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [6]:
# Create model instance
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ANDGateModel().to(device)
print(model)

ANDGateModel(
  (model): Sequential(
    (0): Linear(in_features=2, out_features=1, bias=True)
    (1): Sigmoid()
  )
)


In [7]:
# Loss and optimizer
loss_function = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [8]:
# Training loop
for epoch in range(1000):
    optimizer.zero_grad()
    output = model(X)
    loss = loss_function(output, y)
    loss.backward()
    optimizer.step()

# Prediction
predictions = model(X)
predicted_labels = (predictions > 0.5).float()
print("Predictions:", predicted_labels)

Predictions: tensor([[0.],
        [0.],
        [0.],
        [1.]])


In [10]:
  # Calculate accuracy
correct = (predicted_labels == y).sum().item()
total = y.size(0)
accuracy = correct / total * 100
print(f"Accuracy: {accuracy:.2f}%")    

Accuracy: 100.00%


In [12]:
#Saving the Model
torch.save(model.state_dict(), "AND_gate_model.pth")

In [14]:
#loading the model and testing
loaded_model = ANDGateModel()
loaded_model.load_state_dict(torch.load("AND_gate_model.pth"))
loaded_model.eval()
with torch.no_grad():
    test_output = loaded_model(X)
    predictions = (test_output > 0.5).float()
    print("Predictions after loading model:")
    print(predictions)

Predictions after loading model:
tensor([[0.],
        [0.],
        [0.],
        [1.]])


# BUILDING NEURAL NETWORK FOR XOR GATE

In [15]:
#1. XOR Gate data
X = torch.tensor([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=torch.float32)

y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

In [16]:
#Model building
class XORModel(nn.Module):
    def __init__(self):
        super(XORModel, self).__init__()
        self.linear = nn.Linear(2, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        return self.activation(self.linear(x))

model = XORModel()

In [18]:
#Loss and Optimizer
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [21]:
# Training
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

# Evaluation
with torch.no_grad():
    outputs = model(X)
    predicted = (outputs > 0.5).float()
    accuracy = (predicted == y).sum().item() / y.size(0)
    print("Final Loss:", loss.item())
    print("Accuracy:", accuracy * 100, "%")
    print("Predicted:")
    print(predicted)

Final Loss: 0.6931471824645996
Accuracy: 25.0 %
Predicted:
tensor([[0.],
        [0.],
        [0.],
        [1.]])



# Finding: Why a Single Neuron Fails on XOR

The **XOR problem is not linearly separable**.

A **single neuron** with a **linear boundary** (i.e., a sigmoid activation on a linear combination of inputs) **cannot** correctly classify XOR inputs.

##  Limitations of a Single Neuron

- A single-layer neural network learns only **linear decision boundaries**.
- **XOR** requires a **non-linear decision boundary**, which a single neuron cannot provide.

##  Requirements to Learn XOR

To solve the XOR problem, a neural network must include:

- At least **one hidden layer**
- **Non-linear activation functions** such as:
  - `ReLU`
  - `tanh`
  - `sigmoid` (in a deeper architecture)

In [22]:
#Solving the Problem
class XORNet(nn.Module):
    def __init__(self):
        super(XORNet, self).__init__()
        self.hidden = nn.Linear(2, 2)
        self.output = nn.Linear(2, 1)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.tanh(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

model = XORNet()

In [23]:
# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training
for epoch in range(10000):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

# Evaluation
with torch.no_grad():
    output = model(X)
    predicted = (output > 0.5).float()
    accuracy = (predicted == y).sum().item() / y.size(0)
    print("Final Loss:", loss.item())
    print("Accuracy:", accuracy * 100, "%")
    print("Predictions:\n", predicted)

Final Loss: 0.003473685123026371
Accuracy: 100.0 %
Predictions:
 tensor([[0.],
        [1.],
        [1.],
        [0.]])
